# Notebook 06 — MediaWiki Image Upload

**Project:** Linked Open Exhibition — NFDI4Culture / Hochschule Hannover (BIM-126-02)  
**AI attribution:** GitHub Copilot (Claude Sonnet 4.6)  
**Depends on:**
- `03_dnb_cover_images.ipynb` → `catalogues/images/*.jpg`
- `05_wikibase_upload.ipynb` → items exist in Wikibase
- `.env` file with `MW_URL`, `MW_USER`, `MW_PASSWORD`

**Purpose:** Upload exhibition cover images to the MediaWiki instance and link each uploaded file back to its Wikibase item via the `image` property.

---

## Background: mwclient

`mwclient` is a Python library for interacting with the MediaWiki API. It handles authentication, file uploads, and page edits.

**File naming convention:** `Sprengel_Exhibition_{IDN}.jpg`  
Each uploaded file gets a description page with source, date fetched, and rights notice.

## Cover image rights reminder

> Cover images are publisher-supplied and are **not** CC0. They are uploaded here solely for use within the project Wikibase/MediaWiki instance for educational purposes. Do not redistribute.

In [ ]:
import os, json, time
import pandas as pd
import mwclient
from pathlib import Path
from dotenv import load_dotenv
from datetime import date

load_dotenv(Path("../../.env"))

MW_URL  = os.getenv("MW_URL",  "https://wikibase.wbworkshop.tibwiki.io")
MW_USER = os.getenv("MW_USER")
MW_PASS = os.getenv("MW_PASSWORD")

if not MW_USER or not MW_PASS:
    raise EnvironmentError("Set MW_USER and MW_PASSWORD in your .env file.")

# mwclient expects host without scheme
host = MW_URL.replace("https://", "").replace("http://", "").rstrip("/")
scheme = "https" if MW_URL.startswith("https") else "http"

site = mwclient.Site(host, scheme=scheme, path="/w/")
site.login(MW_USER, MW_PASS)
print(f"Logged in to MediaWiki: {MW_URL}")

## Step 1 — Load data

In [ ]:
CSV_PATH  = Path("../sprengel_exhibitions.csv")
IMAGE_DIR = Path("../images")

df = pd.read_csv(CSV_PATH)
print(f"Records: {len(df)}")

image_files = {p.stem: p for p in IMAGE_DIR.glob("*.jpg")}
print(f"Local images available: {len(image_files)}")

## Step 2 — Upload images to MediaWiki

In [ ]:
TODAY = date.today().isoformat()
uploaded = 0
skipped  = 0

for _, row in df.iterrows():
    idn = str(row.get("idn", "")).strip()

    if idn not in image_files:
        skipped += 1
        continue

    img_path   = image_files[idn]
    file_name  = f"Sprengel_Exhibition_{idn}.jpg"
    title_text = str(row.get("title", "")).strip()[:200]
    year_text  = str(row.get("year", "")).strip()

    description = (
        f"== Summary ==\n"
        f"Exhibition catalogue cover: {title_text} ({year_text})\n\n"
        f"* '''Source''': DNB catalogue enrichment API (https://portal.dnb.de)\n"
        f"* '''DNB IDN''': {idn}\n"
        f"* '''Date fetched''': {TODAY}\n"
        f"* '''Rights''': Publisher-supplied cover image. Not CC0. "
        f"Uploaded for educational/non-commercial use within this project only. "
        f"Do not redistribute.\n"
    )

    try:
        with open(img_path, "rb") as f:
            site.upload(
                file=f,
                filename=file_name,
                description=description,
                ignore=True,  # overwrite if already exists
                comment=f"Bot: upload cover image for DNB IDN {idn}",
            )
        print(f"  Uploaded: {file_name}")
        uploaded += 1
    except Exception as e:
        print(f"  ERROR for {idn}: {e}")

    time.sleep(0.5)

print(f"\nUploaded: {uploaded} | Skipped (no image): {skipped}")

## Step 3 — Link images to Wikibase items

Add an `image` property statement to each Wikibase item pointing to the uploaded MediaWiki file.

In [20]:
import requests as req

SPARQL_URL = os.getenv("SPARQL_URL", "https://query.wbworkshop.tibwiki.io/sparql")

MAP_PATH  = Path("../wikibase_property_map.json")
wbmap     = json.loads(MAP_PATH.read_text(encoding="utf-8"))
P_DNB_IDN = wbmap["properties"]["DNB IDN"]

# wdt: is NOT auto-resolved on this local Wikibase instance — must declare prefix explicitly
WDT_PREFIX = f"{MW_URL.rstrip('/')}/prop/direct/"

def find_item_by_idn(idn):
    """Return QID of an existing item with this DNB IDN via SPARQL, or None."""
    sparql = f"""
        PREFIX wdt: <{WDT_PREFIX}>
        SELECT ?item WHERE {{
          ?item wdt:{P_DNB_IDN} "{idn}" .
        }} LIMIT 1
    """
    try:
        resp = req.get(
            SPARQL_URL,
            params={"query": sparql, "format": "json"},
            headers={"Accept": "application/sparql-results+json"},
            timeout=15,
        )
        bindings = resp.json()["results"]["bindings"]
        if bindings:
            return bindings[0]["item"]["value"].split("/")[-1]
    except Exception:
        pass
    return None

print(f"SPARQL endpoint: {SPARQL_URL}")
print(f"wdt: prefix: {WDT_PREFIX}")


SPARQL endpoint: https://query.wbworkshop.tibwiki.io/sparql
wdt: prefix: https://wikibase.wbworkshop.tibwiki.io/prop/direct/


In [21]:
linked = 0

for _, row in df.iterrows():
    idn = str(row.get("idn", "")).strip()
    file_name = f"Sprengel_Exhibition_{idn}.jpg"

    if idn not in image_files:
        continue

    qid = find_item_by_idn(idn)
    if not qid:
        print(f"  No Wikibase item found for IDN {idn}")
        continue

    if "image" not in wbmap["properties"]:
        print(f"  No 'image' property in map — skipping {qid}")
        linked += 1
        continue

    P_IMAGE = wbmap["properties"]["image"]
    try:
        csrf = site.get_token("csrf")
        site.api(
            "wbcreateclaim",
            entity=qid,
            snaktype="value",
            property=P_IMAGE,
            value=json.dumps(file_name),
            token=csrf,
            summary=f"Bot: link cover image for DNB IDN {idn}",
        )
        print(f"  Linked image to {qid}: {file_name}")
        linked += 1
    except Exception as e:
        print(f"  ERROR for IDN {idn}: {e}")

    time.sleep(0.5)

print(f"\nImage links added: {linked}")


  Linked image to Q6: Sprengel_Exhibition_1375818457.jpg
  Linked image to Q7: Sprengel_Exhibition_1375817957.jpg
  Linked image to Q8: Sprengel_Exhibition_1347131019.jpg
  Linked image to Q9: Sprengel_Exhibition_1357144318.jpg
  Linked image to Q11: Sprengel_Exhibition_1375752529.jpg
  No Wikibase item found for IDN 1375750437
  Linked image to Q13: Sprengel_Exhibition_1362737259.jpg
  Linked image to Q14: Sprengel_Exhibition_1366635671.jpg
  Linked image to Q15: Sprengel_Exhibition_129318635X.jpg
  Linked image to Q16: Sprengel_Exhibition_1325410217.jpg
  Linked image to Q19: Sprengel_Exhibition_1326056700.jpg
  Linked image to Q20: Sprengel_Exhibition_1318695635.jpg
  Linked image to Q21: Sprengel_Exhibition_1315847256.jpg
  Linked image to Q22: Sprengel_Exhibition_1288765711.jpg
  Linked image to Q23: Sprengel_Exhibition_1310094683.jpg
  Linked image to Q24: Sprengel_Exhibition_1277295123.jpg
  Linked image to Q25: Sprengel_Exhibition_1287958486.jpg
  Linked image to Q26: Sprengel_

---

**Pipeline complete.** The exhibitions are now in Wikibase with cover images in MediaWiki.  
Run `quarto render` from the repository root to publish the website.